# F1 2026 Title Odds
## Simulating the Remaining Races from the Weekly Tracker

# Data Preparation
## Imports

In [1]:
import pandas as pd
import numpy as np

## Load Tracker

In [2]:
raw = pd.read_excel('F1_2026_Title_Tracker_v2.xlsx', sheet_name='Weekly_Points_2026', header=None)
print(raw.shape)
print(raw.iloc[0, 0:5].tolist())
print(raw.iloc[1, 0:8].tolist())

(10, 27)
['Race_Number', 'Race_Name', 'Sprint_Weekend', 'K. Antonelli', nan]
[nan, nan, nan, 'Finish_Position', 'Points_Scored', 'Cumulative_Points', 'Gap_to_Leader', 'Finish_Position']


## Reshape to Long Format

In [3]:
drivers = [raw.iloc[0, 3 + 4*k] for k in range(6)]

frames = []
for k, driver in enumerate(drivers):
    start = 3 + 4*k
    block = raw.iloc[3:, [0, 1, 2, start, start + 1]].copy()
    block.columns = ['Race_Number', 'Race_Name', 'Sprint_Weekend', 'Finish_Position', 'Points_Scored']
    block.insert(0, 'Driver', driver)
    frames.append(block)

results = pd.concat(frames, ignore_index=True)
print(results.shape)
results.head(8)

(42, 6)


,Driver,Race_Number,Race_Name,Sprint_Weekend,Finish_Position,Points_Scored
0,K. Antonelli,17,Singapore GP,Yes,3,20
1,K. Antonelli,18,United States GP,No,1,25
2,K. Antonelli,19,Gran Premio De Mexico,No,3,15
3,K. Antonelli,20,Sao Paula GP,No,5,10
4,K. Antonelli,21,Las Vegas GP,No,4,12
5,K. Antonelli,22,Qatar GP,No,NaN,NaN
6,K. Antonelli,23,Abu Dhabi GP,No,NaN,NaN
7,G. Russell,17,Singapore GP,Yes,9,2


In [4]:
print(results.dtypes)

Driver                str
Race_Number        object
Race_Name             str
Sprint_Weekend        str
Finish_Position    object
Points_Scored      object
dtype: object


In [5]:
results['Race_Number'] = pd.to_numeric(results['Race_Number'])
results['Points_Scored'] = pd.to_numeric(results['Points_Scored'], errors='coerce')
results['Finish_Num'] = pd.to_numeric(results['Finish_Position'], errors='coerce')

print(results.dtypes)
print(results[['Points_Scored', 'Finish_Num']].isna().sum())

Driver                 str
Race_Number          int64
Race_Name              str
Sprint_Weekend         str
Finish_Position     object
Points_Scored      float64
Finish_Num         float64
dtype: object
Points_Scored    12
Finish_Num       12
dtype: int64


## Check Against Official Standings

In [6]:
baseline = pd.Series({raw.iloc[0, 3 + 4*k]: raw.iloc[2, 4 + 4*k] for k in range(6)}).astype(float)
print(baseline)

K. Antonelli     320.0
G. Russell       236.0
L. Hamilton      214.0
C. Leclerc       191.0
L. Norris        188.0
M. Verstappen    188.0
dtype: float64


In [7]:
official = pd.read_excel('F1_2026_Title_Tracker_v2.xlsx', sheet_name='Title_Race',
                         usecols='A,E', nrows=6).set_index('Driver')['Official_Latest']

current = baseline + results.groupby('Driver')['Points_Scored'].sum()

check = pd.DataFrame({'Baseline': baseline, 'Computed': current, 'Official': official})
check['Match'] = check['Computed'] == check['Official']
check

,Baseline,Computed,Official,Match
C. Leclerc,191.0,250.0,250,True
G. Russell,236.0,264.0,264,True
K. Antonelli,320.0,402.0,402,True
L. Hamilton,214.0,267.0,267,True
L. Norris,188.0,264.0,264,True
M. Verstappen,188.0,304.0,304,True


# Current Standings
## Points and Gap to Leader

In [8]:
standings = current.sort_values(ascending=False).rename('Points').to_frame()
standings['Gap_to_Leader'] = standings['Points'] - standings['Points'].max()
standings

,Points,Gap_to_Leader
K. Antonelli,402.0,0.0
M. Verstappen,304.0,-98.0
L. Hamilton,267.0,-135.0
G. Russell,264.0,-138.0
L. Norris,264.0,-138.0
C. Leclerc,250.0,-152.0


## Races Left and Points Available

In [9]:
played = results.groupby('Race_Number')['Points_Scored'].apply(lambda s: s.notna().all())
remaining = results.drop_duplicates('Race_Number').loc[lambda d: ~d['Race_Number'].map(played), ['Race_Number', 'Race_Name', 'Sprint_Weekend']]
remaining['Max_Points'] = np.where(remaining['Sprint_Weekend'] == 'Yes', 33, 25)

print(len(remaining), remaining['Max_Points'].sum())
remaining

2 50


,Race_Number,Race_Name,Sprint_Weekend,Max_Points
5,22,Qatar GP,No,25
6,23,Abu Dhabi GP,No,25


## Points Needed to Clinch

In [10]:
next_max = remaining['Max_Points'].iloc[0]
left_after = remaining['Max_Points'].sum() - next_max

rivals = standings.drop('K. Antonelli')
gap = -rivals['Gap_to_Leader']

ant_pts = [25, 18, 15, 12, 10, 8, 6, 4, 2, 1, 0]

need = pd.DataFrame({a: gap + a - left_after - 1 for a in ant_pts})
need.columns.name = 'Antonelli_Points'
need

Antonelli_Points,25,18,15,12,10,8,6,4,2,1,0
M. Verstappen,97.0,90.0,87.0,84.0,82.0,80.0,78.0,76.0,74.0,73.0,72.0
L. Hamilton,134.0,127.0,124.0,121.0,119.0,117.0,115.0,113.0,111.0,110.0,109.0
G. Russell,137.0,130.0,127.0,124.0,122.0,120.0,118.0,116.0,114.0,113.0,112.0
L. Norris,137.0,130.0,127.0,124.0,122.0,120.0,118.0,116.0,114.0,113.0,112.0
C. Leclerc,151.0,144.0,141.0,138.0,136.0,134.0,132.0,130.0,128.0,127.0,126.0


# Race Result Distributions
## Load Races 1-16

In [11]:
hist_raw = pd.read_excel('F1_2026_Championship_Data.xlsx', sheet_name='Race_Results_1_16', header=None)
print(hist_raw.shape)
print(hist_raw.iloc[0, 0:5].tolist())
print(hist_raw.iloc[1, 0:5].tolist())

(18, 15)
['Race_Number', 'Race_Name', 'Sprint_Weekend', 'K. Antonelli', nan]
[nan, nan, nan, 'Finish_Position', 'Points_Scored']


In [12]:
hist_drivers = [hist_raw.iloc[0, 3 + 2*k] for k in range(6)]

hist_frames = []
for k, driver in enumerate(hist_drivers):
    start = 3 + 2*k
    block = hist_raw.iloc[2:, [0, 1, 2, start, start + 1]].copy()
    block.columns = ['Race_Number', 'Race_Name', 'Sprint_Weekend', 'Finish_Position', 'Points_Scored']
    block.insert(0, 'Driver', driver)
    hist_frames.append(block)

history = pd.concat(hist_frames, ignore_index=True)
print(history.shape)
history.head(8)

(96, 6)


,Driver,Race_Number,Race_Name,Sprint_Weekend,Finish_Position,Points_Scored
0,K. Antonelli,1,Australian GP,No,2,18
1,K. Antonelli,2,Chinese GP,Yes,1,29
2,K. Antonelli,3,Japanese GP,No,1,25
3,K. Antonelli,4,Miami GP,Yes,1,28
4,K. Antonelli,5,Canadian GP,Yes,1,31
5,K. Antonelli,6,Monaco GP,No,1,25
6,K. Antonelli,7,Barcelona GP,No,DNF,0
7,K. Antonelli,8,Austrian GP,No,3,15


## Combine All Races

In [13]:
history['Race_Number'] = pd.to_numeric(history['Race_Number'])
history['Points_Scored'] = pd.to_numeric(history['Points_Scored'], errors='coerce')

played = results[results['Points_Scored'].notna()][history.columns]
all_races = pd.concat([history, played], ignore_index=True)

print(all_races.shape)
print(all_races.groupby('Driver')['Points_Scored'].sum())

(126, 6)
Driver
C. Leclerc       250.0
G. Russell       264.0
K. Antonelli     402.0
L. Hamilton      267.0
L. Norris        264.0
M. Verstappen    304.0
Name: Points_Scored, dtype: float64


## Race Points by Driver

In [14]:
race_pts = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}

all_races['Finish_Num'] = pd.to_numeric(all_races['Finish_Position'], errors='coerce')
all_races['Race_Points'] = all_races['Finish_Num'].map(race_pts).fillna(0)

all_races.groupby('Driver')['Race_Points'].agg(['mean', 'sum'])

,mean,sum
Driver,,
C. Leclerc,10.285714,216.0
G. Russell,10.952381,230.0
K. Antonelli,17.666667,371.0
L. Hamilton,11.428571,240.0
L. Norris,11.000000,231.0
M. Verstappen,13.523810,284.0


## DNF Rates

In [15]:
all_races['DNF'] = all_races['Finish_Position'].astype(str).str.upper().isin(['DNF', 'DNS', 'DSQ'])

dnf = all_races.groupby('Driver')['DNF'].agg(['sum', 'mean'])
dnf

,sum,mean
Driver,,
C. Leclerc,3,0.142857
G. Russell,3,0.142857
K. Antonelli,1,0.047619
L. Hamilton,1,0.047619
L. Norris,4,0.190476
M. Verstappen,4,0.190476


# Simulation
## Simulate the Remaining Races

In [16]:
rng = np.random.default_rng(2026)
n_sims = 100_000
n_left = len(remaining)

pool = {d: all_races.loc[all_races['Driver'] == d, 'Race_Points'].values for d in standings.index}

draws = {d: rng.choice(pool[d], size=(n_sims, n_left)) for d in standings.index}
final = pd.DataFrame({d: current[d] + draws[d].sum(axis=1) for d in standings.index})

final.mean().round(1)

K. Antonelli     437.3
M. Verstappen    331.0
L. Hamilton      289.9
G. Russell       285.9
L. Norris        286.0
C. Leclerc       270.6
dtype: float64

## Title Probabilities

In [17]:
winner = final.idxmax(axis=1)
title_prob = winner.value_counts(normalize=True).reindex(standings.index).fillna(0)
(title_prob * 100).round(3)

K. Antonelli     100.0
M. Verstappen      0.0
L. Hamilton        0.0
G. Russell         0.0
L. Norris          0.0
C. Leclerc         0.0
Name: proportion, dtype: float64

## Antonelli Final Points Range

In [18]:
print(final['K. Antonelli'].quantile([0.05, 0.5, 0.95]))

margin = final['K. Antonelli'] - final['G. Russell']
print(margin.quantile([0.001, 0.05, 0.5]))

0.05    417.0
0.50    439.0
0.95    452.0
Name: K. Antonelli, dtype: float64
0.001     98.0
0.050    123.0
0.500    152.0
dtype: float64


## Clinch Timing

In [19]:
cum = {d: draws[d].cumsum(axis=1) + current[d] for d in standings.index}

rivals = [d for d in standings.index if d != 'K. Antonelli']
rival_best = np.max([cum[d] for d in rivals], axis=0)
lead = cum['K. Antonelli'] - rival_best

left_after = remaining['Max_Points'].sum() - remaining['Max_Points'].cumsum().values

clinched = lead > left_after
clinch_by = pd.Series(clinched.mean(axis=0), index=remaining['Race_Name'].values)
(clinch_by * 100).round(1)

Qatar GP        100.0
Abu Dhabi GP    100.0
dtype: float64